In [22]:
import torch

# A

In [23]:
def can_be_expanded(A: torch.Tensor, B: torch.Tensor) -> bool:
  r"""
    Returns True iff A can be expanded into B
  """

  index_A = A.dim() - 1
  index_B = B.dim() - 1

  if index_A > index_B:
    # A has more dimensions than B
    return False

  while index_A >= 0:
    if A.shape[index_A] != 1 and A.shape[index_A] != B.shape[index_B]:
      # Current A's dimension can only be expanded if its length is 1 or its length matches that of B's
      return False

    index_A -= 1
    index_B -= 1
  return True

def my_expand_as(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
  r"""
    Expands A into dimensions of B
  """
  if not can_be_expanded(A, B):
    raise RuntimeError

  # Clone to avoid in-place modification of A
  C = A.clone()

  # Add singleton dimensions at the beginning until ranks match
  while C.dim() < B.dim():
    C = C.unsqueeze(0)

  for dim in range(C.dim()):
    if C.shape[dim] == 1 and B.shape[dim] > 1:
      # Replicate along this axis by concatenating copies
      C = torch.cat([C] * B.shape[dim], dim=dim)

  return C

# B

In [24]:
def can_be_broadcasted(A: torch.Tensor, B: torch.Tensor) -> tuple[bool, torch.Size | None]:
  r"""Determine if tensors A and B can be broadcasted together.

    Returns (True, broadcast_shape) if compatible, otherwise (False, None).
  """

  C = [0] * max(len(A.shape), len(B.shape))

  for i in range(1, min(A.dim(), B.dim()) + 1):
    a, b = A.shape[-i], B.shape[-i]
    if (a != b) and (a != 1) and (b != 1):
      return False, None
    # b may be 0 here, thus not using max()
    C[-i] = a if b == 1 else b

  diff = A.dim() - B.dim()
  if diff != 0:
      longer = A.shape if diff > 0 else B.shape
      C[:abs(diff)] = longer[:abs(diff)]

  return True, torch.Size(C)

# C

In [25]:
def _broadcast_to(t: torch.Tensor, target_shape: torch.Size) -> torch.Tensor:
    res = t.clone()

    # Step 1: Align ranks by prepending singleton dimensions
    while res.dim() < len(target_shape):
        res = res.unsqueeze(0)

    # Step 2: Expand each dimension using torch.cat
    for d, target_size in enumerate(target_shape):
        curr_size = res.shape[d]
        if curr_size == 1 and target_size > 1:
            # Replicate along dimension d
            res = torch.cat([res] * target_size, dim=d)
        elif curr_size == 1 and target_size == 0:
            # Handle size 0 edge case: reshaping empty tensor to target slice
            empty_shape = list(res.shape)
            empty_shape[d] = 0
            res = torch.empty(empty_shape, dtype=res.dtype, device=res.device)
    return res

def my_broadcast(A: torch.Tensor, B: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    possible, size_C = can_be_broadcasted(A, B)

    if not possible:
        raise RuntimeError(f"The size of tensor A {A.shape} must match the size of tensor B {B.shape}")

    result_A = _broadcast_to(A, size_C)
    result_B = _broadcast_to(B, size_C)

    return result_A, result_B


# D

## testing can_be_expanded()

In [26]:
import pytest
from collections.abc import Sequence

@pytest.mark.parametrize(
    ("shape_a", "shape_b", "expected"),
    [
        # Identical shapes
        ((1,), (1,), True),
        ((5, 2), (5, 2), True),
        # Singleton expansions
        ((1,), (5,), True),
        ((1,), (5, 2), True),
        ((1, 2), (5, 2), True),
        ((1, 3), (3, 3), True),
        # Prepending leading singleton/new dimensions
        ((2,), (5, 2), True),
        ((5, 2), (8, 5, 2), True),
        # Incompatible contractions or mismatches
        ((5,), (1,), False),
        ((3,), (5, 2), False),
        ((1, 3), (3, 1), False),
    ],
    ids=[
        "1D_same",
        "2D_same",
        "1D_expand_to_larger",
        "1D_expand_to_2D_singleton",
        "2D_expand_row",
        "2D_expand_col_and_row",
        "1D_prepend_dim_match",
        "2D_prepend_dim_match",
        "1D_cannot_contract",
        "1D_dim_mismatch",
        "2D_shape_transposition_mismatch",
    ],
)

def test_can_be_expanded(
    shape_a: Sequence[int], shape_b: Sequence[int], expected: bool
) -> None:
    a = torch.empty(shape_a)
    b = torch.empty(shape_b)

    assert can_be_expanded(a, b)

## testing my_expand_as()

In [27]:

VALID_EXPAND_CASES = [
    # (input_tensor, target_tensor, expected_output, case_id)
    (
        torch.tensor([1]),
        torch.tensor([3, 2]),
        torch.tensor([1, 1]),
        "1d_singleton_to_1d",
    ),
    (
        torch.tensor([5, 7]),
        torch.tensor([3, 2]),
        torch.tensor([5, 7]),
        "1d_same_shape",
    ),
    (
        torch.tensor([5, 7]),
        torch.tensor([[1, 2], [3, 4]]),
        torch.tensor([[5, 7], [5, 7]]),
        "1d_to_2d_row_broadcast",
    ),
    (
        torch.tensor([1]),
        torch.tensor([[3, 2], [4, 6]]),
        torch.tensor([[1, 1], [1, 1]]),
        "1d_singleton_to_2d",
    ),
    (
        torch.tensor([97, 17, 2]),
        torch.tensor([[11, 12, 13], [21, 22, 23], [31, 32, 33]]),
        torch.tensor([[97, 17, 2], [97, 17, 2], [97, 17, 2]]),
        "1d_to_square_2d",
    ),
    # --- Augmented Cases ---
    (
        torch.tensor(42),
        torch.tensor([[1, 2], [3, 4]]),
        torch.tensor([[42, 42], [42, 42]]),
        "0d_scalar_to_2d",
    ),
    (
        torch.tensor([[1], [2]]),
        torch.tensor([[0, 0, 0], [0, 0, 0]]),
        torch.tensor([[1, 1, 1], [2, 2, 2]]),
        "2d_column_singleton_to_2d",
    ),
    (
        torch.tensor([1, 2]),
        torch.empty(2, 2, 2),
        torch.tensor([[[1, 2], [1, 2]], [[1, 2], [1, 2]]]),
        "1d_to_3d_expansion",
    ),
    (
        torch.empty(0),
        torch.empty(0, 5),
        torch.empty(0, 5),
        "empty_tensor_expansion",
    ),
]


INVALID_EXPAND_CASES = [
    # Incompatible trailing dimension (2 != 3)
    (
        torch.tensor([1, 2]),
        torch.tensor([1, 2, 3]),
        "mismatched_trailing_dim",
    ),
    # Attempting to expand a non-singleton dimension (2 -> 3)
    (
        torch.tensor([[1, 2], [3, 4]]),
        torch.empty(3, 2),
        "incompatible_leading_dim",
    ),
    # Input has more dimensions than target
    (
        torch.tensor([[1, 2]]),
        torch.tensor([1, 2]),
        "input_higher_dim_than_target",
    ),
]


@pytest.mark.parametrize(
    "source, target, expected",
    [case[:3] for case in VALID_EXPAND_CASES],
    ids=[case[3] for case in VALID_EXPAND_CASES],
)
def test_my_expand_as_valid(source: torch.Tensor, target: torch.Tensor, expected: torch.Tensor):
    result = my_expand_as(source, target)

    assert result.shape == target.shape
    torch.testing.assert_close(result, expected)


@pytest.mark.parametrize(
    "source, target",
    [case[:2] for case in INVALID_EXPAND_CASES],
    ids=[case[2] for case in INVALID_EXPAND_CASES],
)
def test_my_expand_as_incompatible_shapes(source: torch.Tensor, target: torch.Tensor):
    with pytest.raises(RuntimeError):
        my_expand_as(source, target)

## testing my_broadcast()

In [ ]:
def test_my_broadcast_valid_cases():
    test_cases = [
        # 1. 1D scalar-like tensor with 2D tensor
        (
            torch.tensor([1]),
            torch.tensor([[11, 12, 13], [21, 22, 23]]),
            torch.tensor([[1, 1, 1], [1, 1, 1]]),
            torch.tensor([[11, 12, 13], [21, 22, 23]]),
        ),
        # 2. 0D scalar tensors
        (
            torch.tensor(5),
            torch.tensor([1, 2, 3]),
            torch.tensor([5, 5, 5]),
            torch.tensor([1, 2, 3]),
        ),
        # 3. Two 0D scalar tensors
        (
            torch.tensor(7),
            torch.tensor(8),
            torch.tensor(7),
            torch.tensor(8),
        ),
        # 4. Identical shapes (no expansion needed)
        (
            torch.tensor([[1, 2], [3, 4]]),
            torch.tensor([[5, 6], [7, 8]]),
            torch.tensor([[1, 2], [3, 4]]),
            torch.tensor([[5, 6], [7, 8]]),
        ),
        # 5. Row vector (1, N) with Column vector (M, 1) -> (M, N)
        (
            torch.tensor([[1, 2, 3]]),
            torch.tensor([[4], [5]]),
            torch.tensor([[1, 2, 3], [1, 2, 3]]),
            torch.tensor([[4, 4, 4], [5, 5, 5]]),
        ),
        # 6. Rank extension: 1D (3,) with 2D (2, 3) -> prepend dimension to (1, 3) then broadcast
        (
            torch.tensor([10, 20, 30]),
            torch.tensor([[1, 2, 3], [4, 5, 6]]),
            torch.tensor([[10, 20, 30], [10, 20, 30]]),
            torch.tensor([[1, 2, 3], [4, 5, 6]]),
        ),
        # 7. Higher dimension expansion: (2, 1, 3) with (1, 4, 1) -> (2, 4, 3)
        (
            torch.arange(6).reshape(2, 1, 3),
            torch.arange(4).reshape(1, 4, 1),
            torch.arange(6).reshape(2, 1, 3).expand(2, 4, 3),
            torch.arange(4).reshape(1, 4, 1).expand(2, 4, 3),
        ),
        # 8. Empty tensor with 0-sized dimension: (0, 1) with (3, 1) -> (0, 1)
        (
            torch.empty((0, 1)),
            torch.tensor([[1], [2], [3]]),
            torch.empty((0, 1)),
            torch.empty((0, 1)),
        ),
    ]

In [ ]:

    for A, B, expected_A, expected_B in test_cases:
        actual_A, actual_B = my_broadcast(A, B)

        # Validate values and shapes match expectations
        torch.testing.assert_close(actual_A

In [28]:
, expected_A)
        torch.testing.assert_close(actual_B, expected_B)

        # Validate consistency with native PyTorch broadcasting logic
        torch_exp_A, torch_exp_B = torch.broadcast_tensors(A, B)
        torch.testing.assert_close(actual_A, torch_exp_A)
        torch.testing.assert_close(actual_B, torch_exp_B)


def test_my_broadcast_invalid_cases():
    invalid_pairs = [
        # Incompatible trailing dimensions: (2,) vs (3,)
        (torch.tensor([1, 2]), torch.tensor([1, 2, 3])),
        # Incompatible inner dimensions: (2, 3) vs (2, 4)
        (torch.zeros(2, 3), torch.zeros(2, 4)),
        # Incompatible leading dimensions after right-alignment: (3, 2) vs (4, 2)
        (torch.zeros(3, 2), torch.zeros(4, 2)),
        # Multi-dimensional mismatch: (2, 3, 4) vs (2, 5, 4)
        (torch.zeros(2, 3, 4), torch.zeros(2, 5, 4)),
    ]

    for A, B in invalid_pairs:
        with pytest.raises(RuntimeError):
            my_broadcast(A, B)